# Week 3 Homework — Multi-Provider LLM Pipeline, Evaluation & Hallucination Audit

In [1]:
!pip install openai anthropic pydantic --quiet

import os, time, json, re, math, requests
import pandas as pd
from google.colab import userdata
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from openai import OpenAI
import anthropic
openai_client    = OpenAI()
anthropic_client = anthropic.Anthropic()

## Part 1 — Multi-Task LLM Pipeline

In [9]:
MODEL_IDS = {"GPT-4o": "gpt-4o", "Claude": "claude-sonnet-4-6"}
# USD per 1M tokens (input, output) - verify against the providers' current pricing pages
PRICES = {"gpt-4o": (2.50, 10.00), "claude-sonnet-4-6": (3.00, 15.00)}

def with_retry(fn, retries=3, base_delay=2.0):
    """Call fn(); on an API error wait and retry with exponential backoff."""
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            print(f"  API error (attempt {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)

def call_gpt(prompt, system=None, json_mode=False):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    def _call():
        t0 = time.time()
        r = openai_client.chat.completions.create(model="gpt-4o", messages=messages, **kwargs)
        return r, time.time() - t0
    r, latency = with_retry(_call)
    return {"text": r.choices[0].message.content, "latency": latency,
            "in_tok": r.usage.prompt_tokens, "out_tok": r.usage.completion_tokens}

def call_claude(prompt, system=None, max_tokens=1500):
    kwargs = {"system": system} if system else {}
    def _call():
        t0 = time.time()
        r = anthropic_client.messages.create(model="claude-sonnet-4-6", max_tokens=max_tokens,
                                             messages=[{"role": "user", "content": prompt}], **kwargs)
        return r, time.time() - t0
    r, latency = with_retry(_call)
    return {"text": r.content[0].text, "latency": latency,
            "in_tok": r.usage.input_tokens, "out_tok": r.usage.output_tokens}

CALLERS = {"GPT-4o": call_gpt, "Claude": call_claude}

def cost(model_id, in_tok, out_tok):
    p_in, p_out = PRICES[model_id]
    return (in_tok * p_in + out_tok * p_out) / 1e6

def parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except Exception:
        return None

In [10]:
# ---- Task A input: a 1,000+ word article (first 1,500 words of a Wikipedia article)
def wiki_text(title):
    r = requests.get("https://en.wikipedia.org/w/api.php",
                     params={"action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "format": "json"},
                     headers={"User-Agent": "genai-course-homework/1.0"}, timeout=30)
    return next(iter(r.json()["query"]["pages"].values()))["extract"]

ARTICLE_A = " ".join(wiki_text("James Webb Space Telescope").split()[:1500])
print("Task A article:", len(ARTICLE_A.split()), "words")

# ---- Task B input: a news article
NEWS_B = """MERIDIAN CITY - Northfield Health Systems announced on 14 March 2025 that it will acquire the
diagnostics start-up Lumina Labs for $420 million, in a deal chief executive Dr. Priya Raman called
"the largest in the company's history." Lumina Labs founder Marcus Webb will join Northfield as
chief scientist once the transaction closes, which is expected in the second quarter of 2026.

Raman said the acquisition would let Northfield offer blood-based screening for early-stage
cancers in all 38 of its hospitals by 2027. Lumina's test has detected stage-one disease in 91
percent of cases in a trial of 12,000 patients, according to a statement released by the company.

State Senator Elena Ortiz welcomed the deal but urged the Federal Trade Commission to review it
carefully. "Patients should not pay more because two health companies became one," Ortiz said
on Friday. Northfield shares rose 6 percent on the news."""

# ---- Task C input: multi-step reasoning problem
# (Check: current profit = 200 x (3.50 - 1.20) - 230 = $230/day; new profit = 170 x (4.00 - 1.20) - 230 = $246/day)
TASK_C = """A bakery sells croissants for $3.50 each. Each croissant costs $1.20 to make, and the bakery has
fixed costs of $230 per day. It currently sells 200 croissants a day. The owner is considering raising the
price to $4.00, which would reduce daily sales by 15%. Should the owner raise the price?
Show your reasoning step by step and give a final recommendation."""

PROMPTS = {
    "A: Summarisation": f"Summarise the following article in exactly 3 sentences.\n\n{ARTICLE_A}",
    "B: Extraction": ("Extract entities from the news article below. Return ONLY a JSON object with keys "
                      "\"people\", \"organisations\", \"dates\" and \"key_claims\" (each a list of strings).\n\n" + NEWS_B),
    "C: Chain-of-thought": TASK_C,
}

Task A article: 1500 words


In [11]:
runs = []   # one entry per (task, model)
for task, prompt in PROMPTS.items():
    for model, caller in CALLERS.items():
        print(f"Running {task} with {model} ...")
        if task.startswith("B") and model == "GPT-4o":
            out = caller(prompt, json_mode=True)
        else:
            out = caller(prompt)
        out.update({"task": task, "model": model,
                    "cost": cost(MODEL_IDS[model], out["in_tok"], out["out_tok"])})
        if task.startswith("B"):
            out["valid_json"] = parse_json(out["text"]) is not None
        runs.append(out)

for r in runs:
    print(f"\n{'=' * 70}\n{r['task']} | {r['model']} | {r['latency']:.2f}s | ${r['cost']:.4f}"
          + (f" | valid JSON: {r['valid_json']}" if "valid_json" in r else ""))
    print(r["text"])

Running A: Summarisation with GPT-4o ...
Running A: Summarisation with Claude ...
Running B: Extraction with GPT-4o ...
Running B: Extraction with Claude ...
Running C: Chain-of-thought with GPT-4o ...
Running C: Chain-of-thought with Claude ...

A: Summarisation | GPT-4o | 2.03s | $0.0070
The James Webb Space Telescope (JWST) is a large and powerful space telescope designed for infrared astronomy, situated at the Sun-Earth L2 point about 1.5 million kilometers from Earth, enabling it to observe cosmic phenomena from the formation of the first galaxies to the characterization of exoplanet atmospheres. Its design includes a large primary mirror made of 18 gold-coated beryllium segments and a sophisticated sunshield made of five layers of Kapton to maintain its operational temperature below 50 K, crucial for detecting faint infrared signals without interference from the telescope's own heat. The project, led by NASA with contributions from the ESA and CSA, endured extensive delays and co

In [12]:
# Fill in after reading the outputs above: (task, model) -> (quality 1-5, key observation)
RATINGS = {}

table1 = pd.DataFrame([{
    "Task": r["task"], "Model": r["model"],
    "Latency (s)": round(r["latency"], 2), "Cost ($)": round(r["cost"], 4),
    "Quality 1-5": RATINGS.get((r["task"], r["model"]), (None, ""))[0],
    "Key observation": RATINGS.get((r["task"], r["model"]), (None, ""))[1],
} for r in runs])
print(table1.to_string(index=False))

               Task  Model  Latency (s)  Cost ($) Quality 1-5 Key observation
   A: Summarisation GPT-4o         2.03    0.0070        None                
   A: Summarisation Claude         5.69    0.0101        None                
      B: Extraction GPT-4o         2.41    0.0030        None                
      B: Extraction Claude         4.00    0.0053        None                
C: Chain-of-thought GPT-4o         3.28    0.0047        None                
C: Chain-of-thought Claude        13.20    0.0123        None                


Analysis (200 words)

Which model would you choose per task and why? Are there tasks where you would use different models in the same pipeline?

Answer:

For Task A (Summarisation), I would choose GPT-4o because it is faster and cheaper than Claude while still producing clear and accurate summaries

For Task B (Extraction), I would choose GPT-4o because it produces structured JSON outputs reliably. It is also faster and less expensive, making it a good choice for extracting information from large amounts of text.

For Task C (Chain-of-thought), I would choose Claude because it provides more detailed explanations. Its responses are easier to follow when a task requires deeper analysis. However, GPT-4o would be a better choice when speed and cost are more important.

Using different models in the same pipeline:

Yes Id say using different models in one pipeline can be useful. I would use GPT-4o to extract important information and numbers from financial articles because it is fast and then I would probably send that information to Claude to produce a more detailed analysis.

So I would choose the model based on the task rather than always using the same one.

## Part 2 — Hallucination Audit
Model: GPT-4o at temperature=0. Domain: machine learning papers and models.

In [18]:
QA = [
    # Easy
    ("Who is Mario's brother?", "Luigi"),
    ("Which company created Minecraft?", "Mojang"),
    ("What is the name of the blue hedgehog from Sega?", "Sonic"),
    ("What is the currency used in Roblox?", "Robux"),
    ("What creature explodes in Minecraft?", "Creeper"),

    # Medium
    ("What is the name of the final boss in Minecraft's End dimension?", "Ender Dragon"),
    ("What is the name of Link's kingdom in The Legend of Zelda?", "Hyrule"),
    ("Which Pokémon evolves from Charmander?", "Charmeleon"),
    ("What is the name of the main character in God of War?", "Kratos"),
    ("Which organization does Master Chief belong to in Halo?", "UNSC"),

    # Difficult
    ("In Minecraft, what is the rare blue ore used to make tools and armor?", "Diamond"),
    ("What year was the original Pokémon Red and Green released in Japan?", "1996"),
    ("What is the name of the fictional kingdom in Super Mario Odyssey where the wedding takes place?", "Moon Kingdom"),
    ("Which company originally developed the first Dark Souls game?", "FromSoftware"),
    ("What is the name of the protagonist in Red Dead Redemption 2?", "Arthur Morgan"),

    # Tricky
    ("What is the name of Mario's third brother?", "Mario has no officially established third brother"),
    ("Which Minecraft mob drops diamonds when defeated?", "No mob normally drops diamonds"),
    ("What is the name of the 2020 Fortnite character called Captain Banana?", "No officially recognized character by that name"),
    ("Which Pokémon evolves directly from Pikachu into Charizard?", "No Pokémon evolves directly from Pikachu into Charizard"),
    ("What is the name of the fifth main character in Among Us?", "Among Us has no fixed set of five main characters")
]
assert len(QA) == 20

def ask(question, system=None):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    def _call():
        return openai_client.chat.completions.create(model="gpt-4o", temperature=0, logprobs=True, messages=messages)
    r = with_retry(_call)
    lps = [t.logprob for t in r.choices[0].logprobs.content]
    return {"answer": r.choices[0].message.content, "conf": math.exp(sum(lps) / len(lps))}

results = []
for i, (q, truth) in enumerate(QA):
    out = ask(q)
    results.append({"q": q, "truth": truth, **out})
    print(f"[{i}] Q: {q}\n    Truth: {truth}\n    Model: {out['answer']}\n    Confidence (mean token prob): {out['conf']:.3f}\n")

[0] Q: Who is Mario's brother?
    Truth: Luigi
    Model: Mario's brother is Luigi. They are both iconic characters from the Nintendo video game franchise.
    Confidence (mean token prob): 0.873

[1] Q: Which company created Minecraft?
    Truth: Mojang
    Model: Minecraft was created by Mojang Studios, a Swedish video game developer. The game was initially developed by Markus Persson, also known as "Notch," who founded Mojang to continue its development.
    Confidence (mean token prob): 0.895

[2] Q: What is the name of the blue hedgehog from Sega?
    Truth: Sonic
    Model: The name of the blue hedgehog from Sega is Sonic.
    Confidence (mean token prob): 0.967

[3] Q: What is the currency used in Roblox?
    Truth: Robux
    Model: The currency used in Roblox is called "Robux." Robux can be used to purchase in-game items, accessories, and other virtual goods within the Roblox platform.
    Confidence (mean token prob): 0.901

[4] Q: What creature explodes in Minecraft?
    Tru

In [19]:
# Grade each response after comparing it with the ground truth above:
#   GRADES[i] = "correct" | "partial" | "hallucinated"
#   HALLU_TYPE[i] = "fabricated citation" | "wrong statistic" | "plausible-but-incorrect claim" | "confident uncertainty avoidance"
GRADES = {
    0: "correct",
    1: "correct",
    2: "correct",
    3: "correct",
    4: "partial",
    5: "correct",
    6: "correct",
    7: "correct",
    8: "correct",
    9: "partial",
    10: "correct",
    11: "correct",
    12: "correct",
    13: "correct",
    14: "correct",
    15: "correct",
    16: "correct",
    17: "correct",
    18: "correct",
    19: "correct"
}
HALLU_TYPE = {}

from collections import Counter
if len(GRADES) < 20:
    print(f"Graded {len(GRADES)}/20 - fill in GRADES for every question index 0-19, then re-run this cell.")
else:
    c = Counter(GRADES.values())
    print(f"Hallucination rate: {c['hallucinated']}/20 hallucinated, {c['partial']}/20 partially correct, {c['correct']}/20 correct")
    print("Taxonomy:", dict(Counter(HALLU_TYPE.get(i, "unlabelled") for i, g in GRADES.items() if g == "hallucinated")))

Hallucination rate: 0/20 hallucinated, 2/20 partially correct, 18/20 correct
Taxonomy: {}


In [20]:
# Mitigation experiment: the 3 highest-confidence hallucinated responses, re-run with an "I don't know" system prompt
SYSTEM_IDK = "If you are not certain, say I don't know."
halluc = [i for i, g in GRADES.items() if g == "hallucinated"]
top3 = sorted(halluc, key=lambda i: -results[i]["conf"])[:3]
if not top3:
    print("No hallucinated responses graded yet.")
for i in top3:
    new = ask(results[i]["q"], system=SYSTEM_IDK)
    print(f"[{i}] Q: {results[i]['q']}\n    Truth:  {results[i]['truth']}")
    print(f"    Before (conf {results[i]['conf']:.3f}): {results[i]['answer']}")
    print(f"    After  (conf {new['conf']:.3f}): {new['answer']}\n")

No hallucinated responses graded yet.


Hallucination analysis (200 words)
What pattern do you see in what the model hallucinated vs what it got right? What does this tell you about the domain-specific risks of deploying this model?

Your answer:

Hallucination Analysis

The model performed well on most gaming questions, especially questions about popular games and well-known characters. It was able to answer basic factual questions correctly. However, it struggled slightly with some more specific details which resulted in partial answers.

This suggests that GPT-4o has strong general gaming knowledge but may be less reliable when answering questions about specific game details or mechanics.

When using AI for gaming information, there is a risk that users may receive incomplete or inaccurate answers, especially about less-known games or updates.

Overall, the model performed well in this test but it is still important not to assume that every answer is correct just because it sounds confident.

## Part 3 — Role-Specific Prompt Comparison
Task: generating test cases for a code function (GPT-4o, same input for all four versions).

In [21]:
FUNCTION_CODE = "\n".join([
    'def is_valid_password(pw: str) -> bool:',
    '    """Valid if it is 8-64 characters long, contains at least one uppercase letter,',
    '    at least one lowercase letter and at least one digit, and contains no spaces."""',
    '    if not 8 <= len(pw) <= 64 or " " in pw:',
    '        return False',
    '    return any(c.isupper() for c in pw) and any(c.islower() for c in pw) and any(c.isdigit() for c in pw)',
])

# Version 1: zero-shot
PROMPT_V1 = f"Write test cases for this Python function:\n\n{FUNCTION_CODE}"

# Version 2: few-shot (exactly 2 examples of the expected input-output format)
PROMPT_V2 = f"""Write test cases for a Python function. Follow the format of the two examples.

Function:
def add(a: int, b: int) -> int:
    return a + b
Test cases:
- name: positive numbers | input: add(2, 3) | expected: 5
- name: negative and positive | input: add(-4, 4) | expected: 0

Function:
def is_even(n: int) -> bool:
    return n % 2 == 0
Test cases:
- name: even number | input: is_even(4) | expected: True
- name: zero is even | input: is_even(0) | expected: True

Function:
{FUNCTION_CODE}
Test cases:"""

# Version 3: chain-of-thought
PROMPT_V3 = (f"Write test cases for this Python function.\n\n{FUNCTION_CODE}\n\n"
             "Think step by step before you answer: first list every rule in the specification, then identify the "
             "boundary values and edge cases for each rule, and only then write the final test cases.")

# Version 4: role in the system prompt + Pydantic output schema
from pydantic import BaseModel

class TestCaseItem(BaseModel):
    name: str
    input: str
    expected: bool
    rationale: str

class TestSuite(BaseModel):
    test_cases: list[TestCaseItem]

SYSTEM_V4 = "You are a senior QA engineer. You write thorough, precise unit test cases that cover boundaries and edge cases."
PROMPT_V4 = f"Write test cases for this Python function:\n\n{FUNCTION_CODE}"

outputs = {}
for v, p in [("Version 1: Zero-shot", PROMPT_V1), ("Version 2: Few-shot", PROMPT_V2), ("Version 3: Chain-of-thought", PROMPT_V3)]:
    outputs[v] = call_gpt(p)["text"]

parse_fn = getattr(openai_client.chat.completions, "parse", None) or openai_client.beta.chat.completions.parse
r = with_retry(lambda: parse_fn(model="gpt-4o",
                                messages=[{"role": "system", "content": SYSTEM_V4}, {"role": "user", "content": PROMPT_V4}],
                                response_format=TestSuite))
suite = r.choices[0].message.parsed
outputs["Version 4: Structured + role"] = suite.model_dump_json(indent=2)

for v, text in outputs.items():
    print(f"\n{'=' * 70}\n{v}\n{'=' * 70}\n{text}")


Version 1: Zero-shot
To test the `is_valid_password` function comprehensively, you should consider various cases that assess the length, presence of uppercase and lowercase letters, digits, spaces, and potential edge cases. Here are some test cases that cover these aspects:

```python
def test_is_valid_password():
    # Test Case 1: Valid password within the constraints
    assert is_valid_password("Valid1Test") == True
    
    # Test Case 2: Valid password with maximum length
    assert is_valid_password("A" * 10 + "b" * 52 + "1") == True
    
    # Test Case 3: Valid password with minimum length
    assert is_valid_password("Aa1bcd12") == True
    
    # Test Case 4: Invalid password, less than 8 characters
    assert is_valid_password("A1bcd") == False
    
    # Test Case 5: Invalid password, more than 64 characters
    assert is_valid_password("A1" + "a" * 63) == False
    
    # Test Case 6: Invalid password, no uppercase letter
    assert is_valid_password("no_uppercase1") == 

In [23]:
# Fill in after reading the outputs: version -> (correctness, format, usefulness, notes), each score 1-5
EVAL = {
    "Version 1: Zero-shot": (
        3, 3, 4,
        "Good variety of tests, but some expected results are incorrect. For example, the 64-character password has no digit and the 64-character boundary test is marked invalid."
    ),

    "Version 2: Few-shot": (
        5, 4, 4,
        "Clear and simple test cases. Covers the main password rules and correctly tests minimum and maximum lengths."
    ),

    "Version 3: Chain-of-thought": (
        5, 4, 5,
        "Most comprehensive version. Covers boundary values, missing requirements, spaces, and multiple failures."
    ),

    "Version 4: Structured + role": (
        3, 5, 4,
        "Very well organized JSON format, but some expected results are incorrect. The password marked as exactly 64 characters is not 64 characters, and the long password has an incorrect expected result."
    )
}

table3 = pd.DataFrame([{
    "Version": v,
    "Correctness": EVAL.get(v, (None, None, None, ""))[0],
    "Format": EVAL.get(v, (None, None, None, ""))[1],
    "Usefulness": EVAL.get(v, (None, None, None, ""))[2],
    "Notes": EVAL.get(v, (None, None, None, ""))[3],
} for v in outputs])
print(table3.to_string(index=False))

                     Version  Correctness  Format  Usefulness                                                                                                                                                                                                Notes
        Version 1: Zero-shot            3       3           4                           Good variety of tests, but some expected results are incorrect. For example, the 64-character password has no digit, and the 64-character boundary test is marked invalid.
         Version 2: Few-shot            5       4           4                                                                                         Clear and simple test cases. Covers the main password rules and correctly tests minimum and maximum lengths.
 Version 3: Chain-of-thought            5       4           5                                                                                             Most comprehensive version. Covers boundary values, missing requireme

Recommendation memo (300 words)

Which prompting strategy would you deploy for this task and why? What failure mode did you observe in the other three?

Answer:

Recommendation Memo

I would choose Version 3 (Chain-of-thought) for this task because it produced the most complete set of password test cases. It checked all the password requirements, including minimum and maximum length, uppercase and lowercase letters, numbers and spaces. It also included edge cases and passwords that failed multiple requirements. This makes it useful for testing whether a password validation function works correctly in different situations.

The other three versions had some weaknesses.

Version 1 (Zero-shot) provided a good variety of test cases but some expected results were incorrect. For example, one password was marked as valid even though it did not contain a digit.

Version 2 (Few-shot) was clear and simple and most of its test cases were correct. However, it included fewer edge cases than Version 3, so it was less comprehensive.

Version 4 (Structured + role) had the best organization because it used JSON with clear names, expected results and explanations. However, some test cases had incorrect expected results, showing that a well-organized format does not always guarantee accuracy.

I would deploy Version 3 because correctness and test coverage are more important than presentation alone but I would still run the generated tests against the actual password function to make sure the expected results are correct.

This comparison shows that prompting strategies can affect the quality of generated test cases. A detailed prompt that explains the requirements and asks for boundary testing can produce more useful results.